# Couro & Asfalto — vídeos de IA das cenas de fim

Gera os vídeos **preso**, **quebrou** e **vitória** com o Wan 2.2 14B (imagem → vídeo) na GPU grátis do Colab.

1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4** → Salvar.
2. Menu **Ambiente de execução → Executar tudo** (ou Ctrl+F9).
3. A 1ª célula baixa ~29 GB de modelos (5–10 min). Cada vídeo leva ~15–20 min (3 vídeos ≈ 1 h).
4. No fim o navegador baixa **videos-cenas.zip** — é só avisar o Claude.

Se faltar memória, o passo 4 troca sozinho para o modelo menor (`Q3_K_M`) e tenta de novo.

In [ ]:
# 1) instala o ComfyUI + nó GGUF e baixa os modelos
QUANT = "Q4_K_M"   # se faltar memória o passo 4 troca sozinho para "Q3_K_M"
import os, subprocess
if not os.path.exists("/content/ComfyUI"):
    subprocess.run("git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI", shell=True, check=True)
    subprocess.run("git clone -q https://github.com/city96/ComfyUI-GGUF /content/ComfyUI/custom_nodes/ComfyUI-GGUF", shell=True, check=True)
    subprocess.run("pip install -q -r /content/ComfyUI/requirements.txt gguf", shell=True, check=True)
from huggingface_hub import hf_hub_download
M = "/content/ComfyUI/models"

def baixar_modelos(quant):
    arquivos = [
        ("QuantStack/Wan2.2-I2V-A14B-GGUF", f"HighNoise/Wan2.2-I2V-A14B-HighNoise-{quant}.gguf", "unet"),
        ("QuantStack/Wan2.2-I2V-A14B-GGUF", f"LowNoise/Wan2.2-I2V-A14B-LowNoise-{quant}.gguf", "unet"),
        ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", "loras"),
        ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", "loras"),
        ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "text_encoders"),
        ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/vae/wan_2.1_vae.safetensors", "vae"),
    ]
    for repo, arq, pasta in arquivos:
        destino = f"{M}/{pasta}/{os.path.basename(arq)}"
        if not os.path.exists(destino):
            caminho = hf_hub_download(repo, arq, local_dir="/content/baixados")
            os.makedirs(os.path.dirname(destino), exist_ok=True)
            os.symlink(caminho, destino)
        print("ok", os.path.basename(arq))

baixar_modelos(QUANT)

In [ ]:
# 2) liga o ComfyUI em segundo plano. --cache-none + --disable-smart-memory: cada etapa solta o
# modelo da memória ao acabar (o Colab grátis tem só 12,7 GB de RAM e os dois modelos não cabem juntos)
import time, urllib.request
comfy = None

def ligar_comfy():
    global comfy
    if comfy is not None and comfy.poll() is None:
        comfy.kill(); comfy.wait()
    log = open("/content/comfy.log", "a")
    comfy = subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--cache-none", "--disable-smart-memory"],
                             cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); print("ComfyUI no ar"); return
        except Exception:
            if comfy.poll() is not None:
                break
            time.sleep(2)
    print(open("/content/comfy.log").read()[-3000:]); raise SystemExit("o ComfyUI não ligou")

subprocess.run("pkill -f 'main.py --listen'", shell=True)
ligar_comfy()

In [ ]:
# 3) as cenas: foto de partida (tirada do próprio jogo), prompt e sementes
import base64
GERAR = ['preso', 'quebrou', 'vitoria']   # tire da lista o que não quiser gerar
VARIACOES = 1   # 2 = duas versões de cada cena (dobra o tempo)
NEGATIVO = 'text, letters, subtitles, watermark, logo, american police car, police text, sheriff, blurry, static, still frame, deformed, extra limbs, extra fingers, melting, morphing, low quality, jpeg artifacts, flicker'
CENAS = {
    'preso': {'prompt': '3D video game cutscene, low-poly style, static camera with a very slow push in. On the shoulder of a Brazilian highway, officers of the Brazilian Federal Highway Police (PRF) in blue caps, sunglasses, black tactical vests, blue shirts and light grey trousers arrest the biker in a black jacket: the officer next to him pulls his hands behind his back and handcuffs him, the other officer walks closer. The dark navy blue PRF SUV with yellow stripes stays parked with its red light bar flashing, red light reflects on the asphalt. The parked red cruiser motorcycle stays still. No new vehicles appear. Afternoon.', 'sementes': [11, 23], 'foto': '...'},
    'quebrou': {'prompt': '3D video game cutscene, low-poly style. A broken red cruiser motorcycle lies on its side on the shoulder of a Brazilian highway, thick grey smoke pours out of the engine, small sparks. The biker in a black jacket limps slowly toward the wrecked motorcycle and puts his hands on his helmet in despair. Camera slowly pushes in. Afternoon.', 'sementes': [11, 23], 'foto': '...'},
    'vitoria': {'prompt': '3D video game cutscene, low-poly style. The biker on the red cruiser motorcycle has just won the race: he keeps his fist raised in the air, pumps it twice in celebration, the motorcycle rolls slowly to a stop. Colorful confetti rains down from the red finish line arch behind him, two rival bikers arrive in the distance. Camera slowly orbits around the winner. Sunny afternoon on a Brazilian highway.', 'sementes': [7, 31], 'foto': '...'},
}
os.makedirs('/content/ComfyUI/input', exist_ok=True)
for n, c in CENAS.items():
    open(f'/content/ComfyUI/input/{n}.jpg', 'wb').write(base64.b64decode(c['foto']))
print('fotos prontas:', list(CENAS))

In [ ]:
# 4) gera os vídeos (832x480, 5 s, 16 quadros/s) e mostra cada um. Cada vídeo roda em DUAS etapas
# (modelo de ruído alto -> latente salvo -> modelo de ruído baixo), uma por vez, para caber na RAM.
import json, glob, uuid, shutil
from IPython.display import HTML, display

class Caiu(Exception):
    pass

def comum(nome, prompt):
    return {
        "7": {"class_type": "CLIPLoader", "inputs": {"clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan"}},
        "8": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["7", 0]}},
        "9": {"class_type": "CLIPTextEncode", "inputs": {"text": NEGATIVO, "clip": ["7", 0]}},
        "10": {"class_type": "VAELoader", "inputs": {"vae_name": "wan_2.1_vae.safetensors"}},
        "11": {"class_type": "LoadImage", "inputs": {"image": f"{nome}.jpg"}},
        "12": {"class_type": "WanImageToVideo", "inputs": {"positive": ["8", 0], "negative": ["9", 0], "vae": ["10", 0], "start_image": ["11", 0], "width": 832, "height": 480, "length": 81, "batch_size": 1}},
    }

def amostrar(modelo, lora, semente, latente, ini, fim, ruido):
    return {
        "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": modelo}},
        "3": {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": lora, "strength_model": 1.0}},
        "5": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["3", 0], "shift": 5.0}},
        "13": {"class_type": "KSamplerAdvanced", "inputs": {"model": ["5", 0], "add_noise": "enable" if ruido else "disable", "noise_seed": semente, "steps": 4, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple", "positive": ["12", 0], "negative": ["12", 1], "latent_image": latente, "start_at_step": ini, "end_at_step": fim, "return_with_leftover_noise": "enable" if ruido else "disable"}},
    }

def etapa_alta(nome, prompt, semente):
    f = comum(nome, prompt)
    f.update(amostrar(f"Wan2.2-I2V-A14B-HighNoise-{QUANT}.gguf", "wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", semente, ["12", 2], 0, 2, True))
    f["14"] = {"class_type": "SaveLatent", "inputs": {"samples": ["13", 0], "filename_prefix": f"latents/{nome}_s{semente}"}}
    return f

def etapa_baixa(nome, prompt, semente, arquivo_latente):
    f = comum(nome, prompt)
    f["20"] = {"class_type": "LoadLatent", "inputs": {"latent": arquivo_latente}}
    f.update(amostrar(f"Wan2.2-I2V-A14B-LowNoise-{QUANT}.gguf", "wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", semente, ["20", 0], 2, 4, False))
    f["15"] = {"class_type": "VAEDecode", "inputs": {"samples": ["13", 0], "vae": ["10", 0]}}
    f["16"] = {"class_type": "SaveImage", "inputs": {"images": ["15", 0], "filename_prefix": f"{nome}_s{semente}"}}
    return f

def pedir(caminho, dados=None):
    req = urllib.request.Request("http://127.0.0.1:8188" + caminho, data=json.dumps(dados).encode() if dados else None, headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req).read())

def rodar(fluxo):
    r = pedir("/prompt", {"prompt": fluxo, "client_id": str(uuid.uuid4())})
    if r.get("node_errors"):
        print(json.dumps(r, indent=1)[-3000:]); raise SystemExit("o ComfyUI recusou o fluxo")
    pid = r["prompt_id"]
    while True:
        if comfy.poll() is not None:
            raise Caiu()
        try:
            h = pedir(f"/history/{pid}")
        except Exception:
            time.sleep(5); continue
        if pid in h and h[pid].get("status", {}).get("completed") is not None:
            break
        time.sleep(5)
    st = h[pid]["status"]
    if st.get("status_str") != "success":
        print(json.dumps(st, indent=1)[-3000:]); print(open("/content/comfy.log").read()[-3000:]); raise SystemExit("erro no ComfyUI")

def gerar_video(nome, c, semente, saida):
    t0 = time.time()
    for velho in glob.glob(f"/content/ComfyUI/output/latents/{nome}_s{semente}_*.latent") + glob.glob(f"/content/ComfyUI/output/{nome}_s{semente}_*.png"):
        os.remove(velho)
    rodar(etapa_alta(nome, c["prompt"], semente))
    lat = sorted(glob.glob(f"/content/ComfyUI/output/latents/{nome}_s{semente}_*.latent"))[-1]
    shutil.copy(lat, f"/content/ComfyUI/input/{os.path.basename(lat)}")
    print(f"  {nome} s{semente}: 1ª etapa em {time.time() - t0:.0f} s")
    rodar(etapa_baixa(nome, c["prompt"], semente, os.path.basename(lat)))
    quadros = f"/content/ComfyUI/output/{nome}_s{semente}_%05d_.png"
    subprocess.run(f"ffmpeg -v error -y -framerate 16 -i {quadros} -c:v libx264 -crf 20 -pix_fmt yuv420p -movflags +faststart {saida}", shell=True, check=True)
    print(f"{nome} semente {semente}: {time.time() - t0:.0f} s")

os.makedirs("/content/videos", exist_ok=True)
for nome in GERAR:
    c = CENAS[nome]
    for semente in c["sementes"][:VARIACOES]:
        saida = f"/content/videos/{nome}_s{semente}.mp4"
        if os.path.exists(saida):
            continue
        for tentativa in range(2):
            try:
                gerar_video(nome, c, semente, saida); break
            except Caiu:
                print(open("/content/comfy.log").read()[-1500:])
                if tentativa or QUANT == "Q3_K_M":
                    raise SystemExit("o ComfyUI caiu de novo (memória) — avise o Claude")
                print("faltou memória: trocando para o modelo Q3_K_M (menor) e tentando de novo")
                QUANT = "Q3_K_M"; baixar_modelos(QUANT); ligar_comfy()
        b64 = base64.b64encode(open(saida, "rb").read()).decode()
        display(HTML(f'<p><b>{nome} — semente {semente}</b></p><video width="640" controls loop autoplay muted src="data:video/mp4;base64,{b64}"></video>'))

In [ ]:
# 5) baixa tudo num zip
shutil.make_archive("/content/videos-cenas", "zip", "/content/videos")
from google.colab import files
files.download("/content/videos-cenas.zip")